# TEC206 Intermediate Programming — Week 7
## Object-Oriented Programming: Polymorphism, Dunder Methods, Abstraction & Encapsulation

**Lecture format:** explanation → demonstration → animation → guided practice → challenge → 30-question quiz  
**Language:** Python 3  
**Prerequisite:** Week 6 inheritance, classes, constructors, methods, and method overriding

---

### Why this workbook exists
Week 6 introduced the four OOP pillars and concentrated on **inheritance**. Week 7 develops the other major ideas in more depth:

1. **Polymorphism** — one interface, many possible behaviours.
2. **Dunder methods** — the special methods that connect our classes to Python syntax and built-in functions.
3. **Abstraction** — defining the essential contract while leaving implementation details to concrete classes.
4. **Encapsulation** — organising state and behaviour and controlling how internal state should be accessed or changed.

The goal is not just to memorise definitions. By the end, you should be able to **read, predict, design, and debug** code that uses these concepts.

> ## How to use this notebook
>
> 1. Run cells from top to bottom.
> 2. Read the explanation **before** running the example.
> 3. Predict the output before executing a cell.
> 4. Change values and add your own objects.
> 5. Use the interactive cells during the lecture.
> 6. Complete the practice exercises before opening the solutions.
>
> ### Interactive requirements
> The animations and quiz use `ipywidgets`. If widgets are not already installed, run:
>
> ```python
> %pip install ipywidgets
> ```
>
> In JupyterLab/modern Jupyter Notebook, widgets normally work after installation. If an interactive cell does not render, the surrounding examples still work as normal Python code.

# 1. Week 6 → Week 7: The Connection

Last week we studied **inheritance**. A child class can reuse and specialise behaviour from a parent class.

The important bridge into Week 7 is **method overriding**:

```text
Parent class defines a method
        ↓
Child class defines a method with the same name
        ↓
The method that runs depends on the actual object
```

That is already one important form of **runtime polymorphism**.

### Four OOP pillars

| Pillar | Main question |
|---|---|
| Encapsulation | How should an object's state be organised and accessed? |
| Abstraction | What behaviour must an object provide, without exposing every implementation detail? |
| Inheritance | How can one class reuse or specialise another class? |
| Polymorphism | How can the same operation work with different object types? |

# 2. Learning Outcomes

By the end of this workbook, you should be able to:

- define **polymorphism** in your own words;
- explain what **runtime** means and why the runtime type of an object matters;
- distinguish **method overriding**, **duck typing**, and polymorphism through operators/built-ins;
- store different object types in one collection and call the same operation safely;
- explain what a **dunder / special method** is;
- use `__str__`, `__repr__`, `__len__`, `__eq__`, and `__add__`;
- explain **operator overloading** and design an appropriate overloaded operator;
- define an **abstract base class** using `ABC` and `@abstractmethod`;
- predict the error produced by an incomplete concrete subclass;
- explain **abstraction** as a design idea, not just a Python decorator;
- distinguish **public**, `_protected-by-convention`, and `__name-mangled` members in Python;
- explain why Python does not provide Java-style private fields;
- write getters and setters using both explicit methods and `@property`;
- combine polymorphism, abstraction, encapsulation, and inheritance in one design.

# 3. Polymorphism

## 3.1 Definition

**Polymorphism** literally means **many forms**.

In object-oriented programming, polymorphism means that the **same operation or interface can produce different behaviour depending on the object involved**.

A useful mental model is:

> **Same message → different object → appropriate response**

For example, we may ask several objects to `make_noise()`. A person, dog, and robot can all respond to that operation differently.

Polymorphism helps us avoid code that constantly asks:

```python
if object_is_a_dog:
    ...
elif object_is_a_robot:
    ...
elif object_is_a_person:
    ...
```

Instead, we can ask each object to perform the behaviour it knows how to perform.

## 3.2 What does **runtime** mean?

**Runtime** is the period when a program is actually executing.

A simplified view is:

```text
Write Python source code
        ↓
Python prepares/compiles code for execution
        ↓
RUNTIME: statements execute, objects are created,
         variables refer to objects, methods are called
        ↓
Program finishes
```

At runtime, a variable name can refer to different objects at different moments.

```python
speaker = Dog()
speaker = Robot()
```

The expression `speaker.make_noise()` is the same in both cases, but Python looks at the **object currently referred to by `speaker`** and resolves the appropriate method.

> **Important:** Python is dynamically typed. A variable does not permanently belong to one class. The **object** has a type; the variable name refers to that object.

In [ ]:
class Dog:
    """A simple class used to demonstrate runtime method selection."""
    def make_noise(self) -> str:
        return "Woof!"


class Robot:
    """Another class that supports the same operation."""
    def make_noise(self) -> str:
        return "Beep boop!"


speaker = Dog()
print(type(speaker).__name__, "->", speaker.make_noise())

speaker = Robot()
print(type(speaker).__name__, "->", speaker.make_noise())

### What happened?

The source expression stayed the same:

```python
speaker.make_noise()
```

but the object stored in `speaker` changed at runtime.

This illustrates **dynamic method dispatch**: the method that runs is determined by the runtime object and Python's attribute/method lookup rules.

## 3.3 Method overriding: polymorphism you already used in Week 6

When a child class defines a method with the same name as a method in its parent class, the child **overrides** the inherited version.

If we call the same method on different child objects, the overridden implementation appropriate to each object runs.

In [ ]:
class Animal:
    """Base class for the overriding example."""
    def make_noise(self) -> str:
        return "Some animal sound"


class Dog(Animal):
    def make_noise(self) -> str:
        return "Woof!"


class Cat(Animal):
    def make_noise(self) -> str:
        return "Meow!"


animals = [Dog(), Cat(), Animal()]

for animal in animals:
    print(f"{type(animal).__name__:>6} -> {animal.make_noise()}")

### Why is this polymorphic?

The loop does not need to know the exact subclass in advance. It simply uses the common operation:

```python
animal.make_noise()
```

The object's runtime type determines which implementation is used.

## 3.4 Python polymorphism does not always require inheritance: duck typing

Python often follows a style called **duck typing**.

The idea is commonly summarised as:

> If an object provides the operation we need, we can use it without first requiring a specific inheritance relationship.

So `Person`, `Dog`, and `Robot` do not need a common parent class for the following loop to work. They only need to provide the method used by the loop.

In [ ]:
class Person:
    def __init__(self, name: str):
        self.name = name

    def make_noise(self) -> str:
        return f"{self.name}: Hello!"


class Dog:
    def __init__(self, name: str):
        self.name = name

    def make_noise(self) -> str:
        return f"{self.name}: Woof!"


class Robot:
    def __init__(self, model: str):
        self.model = model

    def make_noise(self) -> str:
        return f"{self.model}: Beep boop!"


performers = [
    Person("Aisha"),
    Dog("Milo"),
    Robot("RX-7"),
]

for performer in performers:
    print(performer.make_noise())

## 3.5 Why heterogeneous lists are useful

A **heterogeneous collection** contains objects of different types.

```python
performers = [Person(...), Dog(...), Robot(...)]
```

This is useful when all those objects support the operation required by the surrounding code.

### Compare two designs

**Less flexible:**

```python
if isinstance(obj, Person):
    ...
elif isinstance(obj, Dog):
    ...
elif isinstance(obj, Robot):
    ...
```

**More polymorphic:**

```python
obj.make_noise()
```

The second design asks the object to supply its own behaviour. Adding a new compatible class often requires no change to the loop.

> `isinstance()` is still useful when type-specific behaviour is genuinely required. The point is not to ban it; the point is to avoid unnecessary type branching.

# 4. Animation 1 — Runtime Polymorphism Sound Stage

This animation cycles through objects from different classes.

Watch these two things:

1. **The operation stays the same:** `current.make_noise()`
2. **The runtime object changes:** `Person → Dog → Robot → ...`

Use the **Play** control or move the slider manually.

In [ ]:
# Runtime polymorphism animation
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    animated_objects = [
        Person("Aisha"),
        Dog("Milo"),
        Robot("RX-7"),
        Person("Noah"),
        Dog("Luna"),
    ]

    play = widgets.Play(
        value=0,
        min=0,
        max=len(animated_objects) - 1,
        step=1,
        interval=1200,
        description="Play",
    )
    frame = widgets.IntSlider(
        value=0,
        min=0,
        max=len(animated_objects) - 1,
        step=1,
        description="Frame",
        continuous_update=True,
    )
    widgets.jslink((play, "value"), (frame, "value"))
    stage = widgets.Output()

    def draw_stage(change=None):
        current = animated_objects[frame.value]
        with stage:
            stage.clear_output(wait=True)
            display(HTML(f"""
            <div style='border:2px solid #444;border-radius:14px;padding:18px;max-width:720px;'>
              <div style='font-size:14px;opacity:.75;'>Same operation every frame</div>
              <div style='font-family:monospace;font-size:20px;margin:8px 0;'>current.make_noise()</div>
              <hr>
              <div><b>Runtime class:</b> {type(current).__name__}</div>
              <div style='font-size:28px;margin-top:12px;'><b>Response:</b> {current.make_noise()}</div>
            </div>
            """))

    frame.observe(draw_stage, names="value")
    draw_stage()
    display(widgets.HBox([play, frame]), stage)
except Exception as exc:
    print("Interactive widgets are unavailable:", exc)
    print("Fallback demonstration:")
    for current in [Person("Aisha"), Dog("Milo"), Robot("RX-7")]:
        print(type(current).__name__, "->", current.make_noise())

## 4.1 Polymorphism with built-in functions

Polymorphism is not limited to methods that we invent.

The same built-in function can work with many object types:

```python
len("hello")
len([10, 20, 30])
len({"a": 1, "b": 2})
```

The meaning is always related to **length**, but different types provide that behaviour differently.

Later, we will make `len()` work with our own class by defining `__len__`.

In [ ]:
examples = [
    "hello",
    [10, 20, 30],
    {"a": 1, "b": 2},
]

for item in examples:
    print(f"{type(item).__name__:>6} -> len = {len(item)}")

# 5. Dunder Methods / Special Methods

## 5.1 What is a dunder method?

A **dunder method** is a method whose name begins and ends with double underscores:

```python
__init__
__str__
__repr__
__len__
__eq__
__add__
```

"Dunder" is informal shorthand for **double underscore**.

Python's official documentation usually calls these **special methods**.

### Why do special methods matter?

They connect your class to normal Python syntax and built-in operations.

| Python operation | Special method involved |
|---|---|
| Construct an object | `__init__` (after object creation) |
| `str(obj)` / usually `print(obj)` | `__str__` |
| `repr(obj)` | `__repr__` |
| `len(obj)` | `__len__` |
| `a == b` | `__eq__` |
| `a + b` | `__add__` |

This is part of Python's **data model**: classes can define how their instances participate in language operations.

> ### Good practice
>
> In normal application code, prefer the Python operation:
>
> ```python
> len(obj)
> obj1 + obj2
> str(obj)
> ```
>
> rather than explicitly calling `obj.__len__()` or `obj.__add__(other)`.
>
> The special method defines the protocol; the normal syntax communicates the programmer's intention more clearly.

## 5.2 `__str__`: a user-friendly string representation

If a class does not define a helpful string representation, printing an instance usually produces a technical representation.

In [ ]:
class StudentWithoutStr:
    def __init__(self, name: str, student_id: str):
        self.name = name
        self.student_id = student_id


student = StudentWithoutStr("Noah", "S123")
print(student)

Now we add `__str__`.

`__str__` should return a **string** that is useful to a human reader.

In [ ]:
class Student:
    """Represents a student in a simple teaching example."""

    def __init__(self, name: str, student_id: str):
        self.name = name
        self.student_id = student_id

    def __str__(self) -> str:
        """Return a human-readable representation of the student."""
        return f"Student {self.name} ({self.student_id})"


student = Student("Noah", "S123")
print(student)
print(str(student))

## 5.3 `__repr__`: a developer-oriented representation

`__repr__` is intended to provide a useful representation for programmers and debugging.

A common style is to include the class name and important constructor-like values.

In [ ]:
class Student:
    def __init__(self, name: str, student_id: str):
        self.name = name
        self.student_id = student_id

    def __str__(self) -> str:
        return f"Student {self.name} ({self.student_id})"

    def __repr__(self) -> str:
        return f"Student(name={self.name!r}, student_id={self.student_id!r})"


student = Student("Noah", "S123")
print("str :", str(student))
print("repr:", repr(student))

### `!r` inside an f-string

In an f-string, `!r` asks Python to use `repr()` for that value.

```python
f"{self.name!r}"
```

This is useful because strings will normally appear with quotes in a representation.

## 5.4 `__len__`: making `len()` work with our class

`__len__` should return a **non-negative integer** representing the logical size of an object.

In [ ]:
class Playlist:
    """A playlist whose length is the number of stored songs."""

    def __init__(self, songs=None):
        self.songs = list(songs) if songs is not None else []

    def add_song(self, song: str) -> None:
        self.songs.append(song)

    def __len__(self) -> int:
        return len(self.songs)


playlist = Playlist(["Song A", "Song B"])
print("Length:", len(playlist))
playlist.add_song("Song C")
print("Length after adding:", len(playlist))

# 6. Operator Overloading

**Operator overloading** means giving an operator meaningful behaviour for objects of our own class.

For example:

```python
p1 + p2
```

can be defined by implementing `Point.__add__`.

The operator symbol stays the same, but its behaviour depends on the operand types. This is another form of polymorphic behaviour.

> **Design rule:** overload an operator only when the meaning is natural and unsurprising. `Point + Point` is intuitive; `Student + Microwave` probably is not.

## 6.1 `Point` class with `__add__`

For two 2D points/vectors:

```text
(x₁, y₁) + (x₂, y₂) = (x₁ + x₂, y₁ + y₂)
```

In [ ]:
class Point:
    """Represent a point/vector in two-dimensional Cartesian space."""

    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"

    def __repr__(self) -> str:
        return f"Point(x={self.x!r}, y={self.y!r})"

    def __add__(self, other):
        """Return component-wise addition for Point + Point."""
        if not isinstance(other, Point):
            return NotImplemented
        return Point(self.x + other.x, self.y + other.y)


p1 = Point(2, 3)
p2 = Point(5, 7)
p3 = p1 + p2

print("p1 =", p1)
print("p2 =", p2)
print("p1 + p2 =", p3)

### Why return `NotImplemented`?

If `__add__` does not know how to add the other operand type, returning `NotImplemented` tells Python that this implementation does not support that operand combination.

That is generally better protocol behaviour than silently producing nonsense.

If Python cannot find a supported addition implementation, the final result is normally a `TypeError`.

In [ ]:
try:
    print(Point(1, 2) + 10)
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

## 6.2 `__eq__`: defining logical equality

Without a custom equality rule, two separately created objects are not automatically considered equal merely because their attributes happen to contain the same values.

We can define equality for `Point` objects as equality of both coordinates.

In [ ]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Point({self.x!r}, {self.y!r})"

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y


print(Point(2, 3) == Point(2, 3))
print(Point(2, 3) == Point(2, 4))

## 6.3 A domain example: adding groups of animals

Operator overloading does not have to be mathematical, but the meaning should remain clear.

Here, adding two `AnimalGroup` objects combines their members.

In [ ]:
class AnimalGroup:
    def __init__(self, names):
        self.names = list(names)

    def __len__(self) -> int:
        return len(self.names)

    def __str__(self) -> str:
        return ", ".join(self.names)

    def __add__(self, other):
        if not isinstance(other, AnimalGroup):
            return NotImplemented
        return AnimalGroup(self.names + other.names)


pack_a = AnimalGroup(["Milo", "Luna"])
pack_b = AnimalGroup(["Max"])
combined = pack_a + pack_b

print("Pack A:", pack_a)
print("Pack B:", pack_b)
print("Combined:", combined)
print("Number of animals:", len(combined))

# 7. Animation 2 — Which Dunder Method Does Python Use?

This animation steps through common Python syntax and shows the special method associated with the operation.

Use **Play** and focus on the mapping:

```text
Python syntax → special method protocol → class-defined behaviour
```

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    operations = [
        ("print(obj)", "__str__", "Human-readable string form"),
        ("repr(obj)", "__repr__", "Developer-oriented representation"),
        ("len(obj)", "__len__", "Logical size"),
        ("left + right", "__add__", "Addition / combination behaviour"),
        ("left == right", "__eq__", "Logical equality behaviour"),
    ]

    play2 = widgets.Play(value=0, min=0, max=len(operations)-1, interval=1400)
    slider2 = widgets.IntSlider(value=0, min=0, max=len(operations)-1, description="Operation")
    widgets.jslink((play2, "value"), (slider2, "value"))
    out2 = widgets.Output()

    def update_dunder(change=None):
        syntax, method, meaning = operations[slider2.value]
        with out2:
            out2.clear_output(wait=True)
            display(HTML(f"""
            <div style='display:flex;gap:14px;align-items:center;flex-wrap:wrap;'>
              <div style='border:1px solid #777;padding:16px;border-radius:10px;min-width:180px;'>
                <b>Python syntax</b><br><code>{syntax}</code>
              </div>
              <div style='font-size:28px;'>→</div>
              <div style='border:1px solid #777;padding:16px;border-radius:10px;min-width:180px;'>
                <b>Special method</b><br><code>{method}</code>
              </div>
              <div style='font-size:28px;'>→</div>
              <div style='border:1px solid #777;padding:16px;border-radius:10px;min-width:220px;'>
                <b>Meaning</b><br>{meaning}
              </div>
            </div>
            """))

    slider2.observe(update_dunder, names="value")
    update_dunder()
    display(widgets.HBox([play2, slider2]), out2)
except Exception as exc:
    print("Interactive widgets are unavailable:", exc)

# 8. Abstraction

## 8.1 What is abstraction?

**Abstraction** means focusing on the **essential behaviour or interface** while hiding or postponing unnecessary implementation details.

Think about driving a car:

- You use a steering wheel, accelerator, and brake.
- You do not need to manually control every combustion or electrical process.

In software, abstraction lets us say:

> "Every payment method must know how to `pay(amount)`, but different payment types may implement that operation differently."

Abstraction helps keep class hierarchies understandable because it makes the **required contract** explicit.

## 8.2 Abstract Base Classes (ABCs)

Python's `abc` module provides tools for defining **abstract base classes**.

We commonly use:

```python
from abc import ABC, abstractmethod
```

- `ABC` marks a class as participating in the abstract-base-class mechanism.
- `@abstractmethod` marks a method that concrete subclasses are required to implement before they can be instantiated.

An abstract base class can still contain normal methods and shared implementation.

In [ ]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):
    """Abstract contract for a payment method."""

    @abstractmethod
    def pay(self, amount: float) -> str:
        """Process a payment and return a status message."""
        pass


class CreditCardPayment(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"Paid ${amount:.2f} by credit card"


class PayPalPayment(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"Paid ${amount:.2f} using PayPal"


methods = [CreditCardPayment(), PayPalPayment()]
for method in methods:
    print(method.pay(49.95))

### Notice the design

`PaymentMethod` says **what must exist**:

```python
pay(amount)
```

The concrete subclasses decide **how it works**.

This combines:

- **abstraction** — a common required contract;
- **inheritance** — subclasses derive from `PaymentMethod`;
- **polymorphism** — the same `pay()` call produces different behaviour.

## 8.3 What if a subclass does not implement the abstract method?

If a subclass remains incomplete, Python treats it as abstract and prevents instantiation.

The class definition itself can exist. The error occurs when we try to instantiate an incomplete class.

In [ ]:
class BrokenPayment(PaymentMethod):
    # pay() is intentionally missing
    pass


try:
    broken = BrokenPayment()
except TypeError as exc:
    print("Instantiation failed as expected")
    print(type(exc).__name__ + ":", exc)

### Important distinction: `@abstractmethod` vs `raise NotImplementedError`

These ideas are related but not identical.

```python
@abstractmethod
def pay(self, amount):
    ...
```

participates in the ABC mechanism and can prevent incomplete subclasses from being instantiated.

By contrast:

```python
def pay(self, amount):
    raise NotImplementedError
```

is an ordinary concrete method that raises an exception **only if it is called**. It does not by itself make the class abstract.

In [ ]:
class InformalPayment:
    def pay(self, amount):
        raise NotImplementedError("Subclass should provide pay()")


example = InformalPayment()       # Instantiation succeeds
print("Object created:", type(example).__name__)

try:
    example.pay(10)
except NotImplementedError as exc:
    print(type(exc).__name__ + ":", exc)

## 8.4 Abstract methods can still contain implementation

A subtle but important Python feature: an abstract method may contain code.

A subclass must still provide an implementation before becoming concrete, but it can call the abstract base implementation using `super()` when that design is useful.

This is an advanced technique; for this course, remember that **abstract does not necessarily mean empty**.

In [ ]:
class Report(ABC):
    @abstractmethod
    def generate(self) -> str:
        # Shared starting text even though the method is abstract
        return "Report header"


class SalesReport(Report):
    def generate(self) -> str:
        return super().generate() + "\nSales data goes here"


print(SalesReport().generate())

# 9. Abstraction Visual — Contract vs Implementation

The next interactive example lets you switch between concrete subclasses while keeping the same abstract contract.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    choices = {
        "Credit card": CreditCardPayment(),
        "PayPal": PayPalPayment(),
    }

    chooser = widgets.Dropdown(options=list(choices), description="Payment:")
    amount = widgets.FloatSlider(value=25, min=1, max=100, step=1, description="Amount:")
    abstract_out = widgets.Output()

    def show_contract(change=None):
        obj = choices[chooser.value]
        with abstract_out:
            abstract_out.clear_output(wait=True)
            display(HTML(f"""
            <div style='border:1px solid #777;border-radius:12px;padding:16px;max-width:700px;'>
              <b>Abstract contract:</b> <code>pay(amount)</code><br><br>
              <b>Concrete runtime class:</b> {type(obj).__name__}<br>
              <b>Result:</b> {obj.pay(amount.value)}
            </div>
            """))

    chooser.observe(show_contract, names="value")
    amount.observe(show_contract, names="value")
    show_contract()
    display(widgets.VBox([chooser, amount, abstract_out]))
except Exception as exc:
    print("Interactive widgets are unavailable:", exc)

# 10. Encapsulation

## 10.1 What is encapsulation?

**Encapsulation** means bundling an object's **state** and the **methods that operate on that state** inside a class, while defining a sensible way for other code to interact with the object.

Encapsulation is not simply "make everything private".

Good encapsulation aims to:

- keep related state and behaviour together;
- prevent accidental invalid states;
- expose a clear public interface;
- signal which details are internal implementation details;
- allow implementation details to change without breaking user code unnecessarily.

## 10.2 Public, protected-by-convention, and private-style names in Python

Python's conventions differ from languages such as Java or C++.

| Form | Example | Meaning in normal Python practice |
|---|---|---|
| Public | `self.balance` | Intended for normal external use |
| Protected-by-convention | `self._balance` | Internal/non-public API; external access is possible but discouraged unless you know what you are doing |
| Name-mangled / private-style | `self.__pin` | Python rewrites the name to reduce accidental access/overriding; it is **not a security boundary** |

### Critical point

A leading underscore is a **convention**, not an access-control mechanism.

A double-leading-underscore name triggers **name mangling**. It makes accidental access and accidental subclass clashes harder, but determined code can still access the mangled name.

In [ ]:
class AccountDemo:
    def __init__(self):
        self.owner = "Aisha"        # public
        self._balance = 100.0       # non-public by convention
        self.__pin = "4321"         # name-mangled


account = AccountDemo()

print("Public:", account.owner)
print("_balance is still technically accessible:", account._balance)

try:
    print(account.__pin)
except AttributeError as exc:
    print("Direct __pin access failed:", type(exc).__name__)

print("Attributes stored on the object:")
print(account.__dict__)

## 10.3 What exactly is name mangling?

Inside a class named `AccountDemo`, a name such as:

```python
self.__pin
```

is rewritten to a form similar to:

```python
self._AccountDemo__pin
```

This is designed mainly to reduce **accidental name collisions**, especially in subclasses.

It does **not** provide cryptographic privacy or strong access restriction.

In [ ]:
print("Mangled value:", account._AccountDemo__pin)

# This works, which is exactly why double-underscore names should not be
# described as a security mechanism.

## 10.4 Encapsulation through behaviour: a better bank account

A bank account should not allow arbitrary code to create impossible state.

Instead of encouraging this:

```python
account.balance = -1_000_000
```

we can expose meaningful operations such as:

```python
account.deposit(100)
account.withdraw(50)
```

Those methods can validate the requested state change.

In [ ]:
class BankAccount:
    """A small example of encapsulating account state and validation."""

    def __init__(self, owner: str, opening_balance: float = 0.0):
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self.owner = owner
        self._balance = float(opening_balance)

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

    def get_balance(self) -> float:
        return self._balance


account = BankAccount("Aisha", 100)
account.deposit(50)
account.withdraw(30)
print("Balance:", account.get_balance())

# 11. Getters and Setters

A **getter** retrieves a value. A **setter** changes a value, usually with validation or other control.

Getters and setters are useful when direct assignment would allow invalid state or when additional behaviour is required.

Python supports both:

1. explicit methods such as `get_temperature()` and `set_temperature()`;
2. properties using `@property`, which provide controlled access with attribute-like syntax.

## 11.1 Traditional getter and setter methods

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self._celsius = 0.0
        self.set_celsius(celsius)

    def get_celsius(self) -> float:
        return self._celsius

    def set_celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError("Temperature cannot be below absolute zero")
        self._celsius = float(value)


t = Temperature(20)
print(t.get_celsius())
t.set_celsius(25)
print(t.get_celsius())

## 11.2 Pythonic getters and setters with `@property`

A property lets the caller write:

```python
t.celsius

t.celsius = 25
```

while still running getter/setter logic behind the scenes.

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self._celsius = 0.0
        self.celsius = celsius       # Uses the property setter

    @property
    def celsius(self) -> float:
        """Return the temperature in degrees Celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        """Set Celsius after validating physical lower bound."""
        if value < -273.15:
            raise ValueError("Temperature cannot be below absolute zero")
        self._celsius = float(value)


t = Temperature(20)
print("Initial:", t.celsius)
t.celsius = 25
print("Updated:", t.celsius)

try:
    t.celsius = -300
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)

## 11.3 Read-only computed properties

A property does not need a setter.

Here, `fahrenheit` is calculated from `celsius`; callers can read it but there is no direct `fahrenheit` setter.

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self._celsius = float(celsius)

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError("Temperature cannot be below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self) -> float:
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print("Celsius   :", t.celsius)
print("Fahrenheit:", t.fahrenheit)

# 12. Interactive Encapsulation Demonstration

Use the buttons to change the account through its **public behaviour**. The widget intentionally does not mutate `_balance` directly.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    demo_account = BankAccount("Interactive User", 100)
    amount_box = widgets.FloatText(value=10.0, description="Amount")
    deposit_button = widgets.Button(description="Deposit")
    withdraw_button = widgets.Button(description="Withdraw")
    account_out = widgets.Output()

    def redraw_account(message=""):
        with account_out:
            account_out.clear_output(wait=True)
            print(f"Owner   : {demo_account.owner}")
            print(f"Balance : ${demo_account.get_balance():.2f}")
            if message:
                print(message)

    def do_deposit(_):
        try:
            demo_account.deposit(amount_box.value)
            redraw_account("Deposit accepted")
        except ValueError as exc:
            redraw_account("Rejected: " + str(exc))

    def do_withdraw(_):
        try:
            demo_account.withdraw(amount_box.value)
            redraw_account("Withdrawal accepted")
        except ValueError as exc:
            redraw_account("Rejected: " + str(exc))

    deposit_button.on_click(do_deposit)
    withdraw_button.on_click(do_withdraw)
    redraw_account()
    display(widgets.HBox([amount_box, deposit_button, withdraw_button]), account_out)
except Exception as exc:
    print("Interactive widgets are unavailable:", exc)

# 13. Putting All Four OOP Pillars Together

The following example models notifications.

We will use:

- **Abstraction:** `Notification` defines the required `send()` contract.
- **Inheritance:** `EmailNotification` and `SMSNotification` inherit from `Notification`.
- **Polymorphism:** one loop calls `send()` on different concrete objects.
- **Encapsulation:** recipient information is stored inside each object and validated through constructors/properties.

In [ ]:
from abc import ABC, abstractmethod


class Notification(ABC):
    """Abstract base class for notification channels."""

    @abstractmethod
    def send(self, message: str) -> str:
        """Send a message and return a human-readable status."""
        pass


class EmailNotification(Notification):
    def __init__(self, email: str):
        if "@" not in email:
            raise ValueError("A simple valid-looking email address is required")
        self._email = email

    @property
    def destination(self) -> str:
        return self._email

    def send(self, message: str) -> str:
        return f"EMAIL to {self._email}: {message}"


class SMSNotification(Notification):
    def __init__(self, phone: str):
        if not phone.strip():
            raise ValueError("Phone number cannot be empty")
        self._phone = phone

    @property
    def destination(self) -> str:
        return self._phone

    def send(self, message: str) -> str:
        return f"SMS to {self._phone}: {message}"


channels = [
    EmailNotification("student@example.com"),
    SMSNotification("0400 000 000"),
]

for channel in channels:
    print(channel.send("Week 7 starts now"))

## 13.1 Identify each concept

For the code above, answer these before reading on:

1. What is the abstract class?
2. Which method forms the shared contract?
3. Where is inheritance used?
4. Where is polymorphism visible?
5. Which attributes are treated as implementation details?
6. Why are the subclasses concrete rather than abstract?

<details>
<summary><b>Show discussion</b></summary>

1. `Notification` is the abstract class.
2. `send(message)` is the required operation.
3. `EmailNotification(Notification)` and `SMSNotification(Notification)` use inheritance.
4. The loop calls `channel.send(...)` without separate code for each concrete class.
5. `_email` and `_phone` are non-public by convention.
6. Each subclass implements the abstract `send()` method.

</details>

# 14. Common Misunderstandings and Exam Traps

### Misunderstanding 1: "Polymorphism means inheritance"
Not necessarily. Inheritance-based overriding is one route, but Python also commonly uses duck typing and protocol-based behaviour.

### Misunderstanding 2: "The variable has a permanent class"
Python variable names refer to objects. A name may refer to a `Dog` now and a `Robot` later.

### Misunderstanding 3: "A dunder method is any method with underscores"
No. Special methods use specific double-leading and double-trailing names such as `__len__`.

### Misunderstanding 4: "`print()` always calls `__repr__`"
For a normal object, `print(obj)` converts the object to text using `str(obj)`, which uses `__str__` when defined. Representation rules have fallback details, but `__str__` is the intended human-readable hook.

### Misunderstanding 5: "`@abstractmethod` means the method must contain only `pass`"
No. Abstract methods can contain implementation, although subclasses still need to override abstract methods before becoming concrete.

### Misunderstanding 6: "`_balance` cannot be accessed outside the class"
It can. The single underscore is a convention signalling non-public/internal use.

### Misunderstanding 7: "`__pin` is secure/private data"
No. Double-leading-underscore names are name-mangled, not secured.

### Misunderstanding 8: "Every attribute needs a getter and setter"
No. Use properties or controlled methods when they provide a real design benefit such as validation, computation, compatibility, or read-only access.

### Misunderstanding 9: "Operator overloading means changing Python's global `+` operator"
No. A class defines how its own instances participate in the operator protocol.

### Misunderstanding 10: "Returning `NotImplemented` is the same as raising `NotImplementedError`"
No. `NotImplemented` is a special return value used by some binary-operation protocols; `NotImplementedError` is an exception.

# 15. Practice Exercise 1 — Runtime Polymorphism

Create three classes:

- `Person`
- `Dog`
- `Robot`

Each must implement:

```python
move()
```

Store one object from each class in a list and loop over the list, printing the result of `move()`.

### Requirements
- Do not use `if type(...)` or `isinstance(...)` in the loop.
- Each class must return a different message.

In [ ]:
# TODO: Write your solution here.

<details>
<summary><b>Show solution</b></summary>

```python
class Person:
    def move(self):
        return "Person walks"

class Dog:
    def move(self):
        return "Dog runs"

class Robot:
    def move(self):
        return "Robot rolls"

objects = [Person(), Dog(), Robot()]

for obj in objects:
    print(obj.move())
```

The loop is polymorphic because it uses one operation, `move()`, with multiple object types.

</details>

# 16. Practice Exercise 2 — `Book` Dunder Methods

Create a `Book` class with:

- `title`
- `author`
- `pages`
- `__str__` → return a friendly description
- `__repr__` → return a developer-oriented representation
- `__len__` → return the number of pages

Then test:

```python
print(book)
repr(book)
len(book)
```

In [ ]:
# TODO: Write your solution here.

<details>
<summary><b>Show solution</b></summary>

```python
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __str__(self):
        return f"{self.title} by {self.author}"

    def __repr__(self):
        return f"Book(title={self.title!r}, author={self.author!r}, pages={self.pages!r})"

    def __len__(self):
        return self.pages
```

</details>

# 17. Practice Exercise 3 — Operator Overloading

Extend a `Point` class so that:

```python
Point(1, 2) + Point(3, 4)
```

produces a new point representing `(4, 6)`.

Also define logical equality so that:

```python
Point(4, 6) == Point(4, 6)
```

is `True`.

In [ ]:
# TODO: Write your solution here.

<details>
<summary><b>Show solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return Point(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"
```

</details>

# 18. Practice Exercise 4 — Abstraction

Create an abstract class called `Shape` with an abstract method:

```python
area()
```

Then create:

- `Rectangle(width, height)`
- `Circle(radius)`

Each subclass must implement `area()`.

Finally, place a rectangle and circle in a list and call `area()` polymorphically.

In [ ]:
# TODO: Write your solution here.

<details>
<summary><b>Show solution</b></summary>

```python
from abc import ABC, abstractmethod
from math import pi

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return pi * self.radius ** 2

shapes = [Rectangle(3, 4), Circle(2)]
for shape in shapes:
    print(shape.area())
```

</details>

# 19. Practice Exercise 5 — Encapsulation and Properties

Create a class `ExamScore` that stores a score between 0 and 100.

Requirements:

- Store the internal value as `_score`.
- Provide a `score` property.
- Reject values below 0 or above 100 with `ValueError`.
- Add a read-only property `passed` that returns `True` when the score is at least 50.

In [ ]:
# TODO: Write your solution here.

<details>
<summary><b>Show solution</b></summary>

```python
class ExamScore:
    def __init__(self, score):
        self._score = 0
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if not 0 <= value <= 100:
            raise ValueError("Score must be between 0 and 100")
        self._score = value

    @property
    def passed(self):
        return self._score >= 50
```

</details>

# 20. Mini Design Challenge — Transport System

Design a small transport system.

### Requirements

1. Create an abstract class `Vehicle`.
2. Give it an abstract method `move()`.
3. Create at least three concrete subclasses, for example:
   - `Car`
   - `Bicycle`
   - `Train`
4. Each subclass must implement `move()` differently.
5. Store the vehicle name in a `_name` attribute.
6. Expose the name with a read-only `name` property.
7. Implement `__str__` for a friendly description.
8. Store multiple vehicles in a list and call `move()` in one loop.

### Extension
Add a `speed` property that rejects negative speed.

In [ ]:
# TODO: Build the complete mini design here.

# 21. Concept Check — Predict Before You Run

For each example, predict the output or error before executing it.

In [ ]:
class A:
    def speak(self):
        return "A"

class B(A):
    def speak(self):
        return "B"

obj = B()
print(obj.speak())

In [ ]:
class Box:
    def __init__(self, items):
        self.items = list(items)

    def __len__(self):
        return len(self.items)

box = Box([1, 2, 3, 4])
print(len(box))

In [ ]:
from abc import ABC, abstractmethod

class Base(ABC):
    @abstractmethod
    def work(self):
        pass

class Child(Base):
    pass

try:
    Child()
except TypeError as exc:
    print(type(exc).__name__)

# 22. 30-Question Interactive Quiz

This quiz covers the complete Week 7 workbook.

### Features
- **30 questions**
- one question at a time
- four answer choices
- instant feedback
- explanation after checking
- previous/next navigation
- live score
- restart button

The questions test understanding, not just memorisation.

In [ ]:
# Quiz data: 30 questions
quiz_questions = [
    {
        "q": "1. Which statement best defines polymorphism in OOP?",
        "options": [
            "Every class must inherit from exactly one parent",
            "The same operation can produce type-appropriate behaviour for different objects",
            "All object attributes must be private",
            "A class may contain only one method",
        ],
        "answer": 1,
        "explanation": "Polymorphism allows a common operation/interface to be used with different object types, with behaviour appropriate to each object."
    },
    {
        "q": "2. What does runtime refer to?",
        "options": [
            "Only the time spent writing source code",
            "The period while the program is executing",
            "Only the moment a file is saved",
            "A special type of Python class",
        ],
        "answer": 1,
        "explanation": "Runtime is the period during which program instructions are being executed and objects/method calls actually exist."
    },
    {
        "q": "3. Why can `speaker.make_noise()` call different implementations at different times?",
        "options": [
            "Python randomly chooses a method",
            "The object currently referenced by `speaker` can have a different runtime type",
            "Method names are ignored in Python",
            "Every method has the same implementation",
        ],
        "answer": 1,
        "explanation": "The same variable name can refer to different objects. Method lookup depends on the runtime object."
    },
    {
        "q": "4. Which Week 6 concept is already a common form of runtime polymorphism?",
        "options": ["Method overriding", "Comments", "Importing modules", "Integer division"],
        "answer": 0,
        "explanation": "Overridden methods let different subclass objects respond differently to the same method call."
    },
    {
        "q": "5. What best describes duck typing in Python?",
        "options": [
            "An object is usable when it provides the required behaviour, even without a specific inheritance relationship",
            "Every compatible object must inherit from a class named Duck",
            "Python converts every object to the same type",
            "Only built-in classes support polymorphism",
        ],
        "answer": 0,
        "explanation": "Duck typing focuses on whether an object supports the operations required by the code."
    },
    {
        "q": "6. A list contains `Person()`, `Dog()`, and `Robot()` and each implements `move()`. Which loop is the most directly polymorphic?",
        "options": [
            "`for x in items: print(x.move())`",
            "`for x in items: print(type(x))`",
            "`for x in items: print(id(x))`",
            "`for x in items: print(x.__dict__)`",
        ],
        "answer": 0,
        "explanation": "The same `move()` operation is sent to each object and each class supplies its own behaviour."
    },
    {
        "q": "7. What is a dunder method?",
        "options": [
            "A method whose official special name has double underscores at both ends",
            "Any method containing at least one underscore",
            "A method that cannot be called",
            "A method that exists only in abstract classes",
        ],
        "answer": 0,
        "explanation": "Names such as `__str__` and `__len__` are special methods; 'dunder' informally means double underscore."
    },
    {
        "q": "8. Which special method defines the human-friendly string returned by `str(obj)`?",
        "options": ["`__len__`", "`__str__`", "`__eq__`", "`__add__`"],
        "answer": 1,
        "explanation": "`__str__` is the special method intended for a human-readable string form."
    },
    {
        "q": "9. Which special method is intended for a developer-oriented representation?",
        "options": ["`__repr__`", "`__init__`", "`__len__`", "`__add__`"],
        "answer": 0,
        "explanation": "`__repr__` provides a representation useful for debugging/development."
    },
    {
        "q": "10. Which method allows `len(my_object)` to work for a custom class?",
        "options": ["`__size__`", "`__count__`", "`__len__`", "`__length__`"],
        "answer": 2,
        "explanation": "The `len()` protocol uses `__len__`."
    },
    {
        "q": "11. Which special method is associated with `left + right`?",
        "options": ["`__sum__`", "`__plus__`", "`__add__`", "`__eq__`"],
        "answer": 2,
        "explanation": "Custom addition behaviour is defined using `__add__`."
    },
    {
        "q": "12. Which special method is associated with `left == right`?",
        "options": ["`__compare__`", "`__same__`", "`__eq__`", "`__equals__`"],
        "answer": 2,
        "explanation": "Logical equality is defined through `__eq__`."
    },
    {
        "q": "13. What is operator overloading?",
        "options": [
            "Defining meaningful operator behaviour for instances of a class",
            "Replacing Python's `+` operator globally",
            "Using too many operators in one expression",
            "Converting every operator into a function named `operator`",
        ],
        "answer": 0,
        "explanation": "Operator overloading lets a class define how its instances participate in operators such as `+` or `==`."
    },
    {
        "q": "14. In a `Point.__add__` method, why might unsupported operand types return `NotImplemented`?",
        "options": [
            "To tell Python this implementation does not support that operand combination",
            "To immediately terminate the Python interpreter",
            "To create a new abstract class",
            "To convert the other object into a string",
        ],
        "answer": 0,
        "explanation": "`NotImplemented` is a special protocol return value that indicates the operation is unsupported for that operand pairing."
    },
    {
        "q": "15. Which statement about `NotImplemented` and `NotImplementedError` is correct?",
        "options": [
            "They are exactly the same object",
            "`NotImplemented` is a special value; `NotImplementedError` is an exception",
            "Both are decorators",
            "Both automatically make a class abstract",
        ],
        "answer": 1,
        "explanation": "They serve different purposes: one is a special value used by protocols, the other is an exception class."
    },
    {
        "q": "16. What is the main design idea behind abstraction?",
        "options": [
            "Expose essential behaviour while separating/hiding unnecessary implementation detail",
            "Make every method exactly one line long",
            "Remove all classes from a program",
            "Store every variable globally",
        ],
        "answer": 0,
        "explanation": "Abstraction focuses users of a component on what it provides rather than every detail of how it is implemented."
    },
    {
        "q": "17. Which import is commonly used to create an abstract base class in Python?",
        "options": [
            "`from abc import ABC, abstractmethod`",
            "`from oop import Private, Public`",
            "`from runtime import polymorphism`",
            "`from types import constructor`",
        ],
        "answer": 0,
        "explanation": "The standard `abc` module provides `ABC` and `abstractmethod`."
    },
    {
        "q": "18. What does `@abstractmethod` mean for a regular subclass that has not implemented that method?",
        "options": [
            "The incomplete subclass generally cannot be instantiated",
            "The missing method is silently deleted",
            "The method automatically returns zero",
            "The subclass becomes a list",
        ],
        "answer": 0,
        "explanation": "A class using the ABC mechanism cannot be instantiated while required abstract methods remain unimplemented."
    },
    {
        "q": "19. Can an abstract method in Python contain implementation code?",
        "options": [
            "Yes, although concrete subclasses still need to satisfy the abstract-method requirement",
            "No, Python syntax forbids any code in it",
            "Only when the class has no subclasses",
            "Only if the method is also `__len__`",
        ],
        "answer": 0,
        "explanation": "Python abstract methods may contain implementation and can be called via `super()` from an override."
    },
    {
        "q": "20. How does `raise NotImplementedError` differ from `@abstractmethod`?",
        "options": [
            "An ordinary method raising `NotImplementedError` can still exist in an instantiable class; `@abstractmethod` participates in ABC instantiation checks",
            "There is no difference",
            "`NotImplementedError` is used only for properties",
            "`@abstractmethod` is used only for arithmetic",
        ],
        "answer": 0,
        "explanation": "Raising the exception happens when that method runs; `@abstractmethod` can prevent incomplete ABC-derived classes from being instantiated."
    },
    {
        "q": "21. What best defines encapsulation?",
        "options": [
            "Bundling state with behaviour and exposing a controlled, meaningful interface",
            "Making all attributes globally accessible",
            "Using inheritance without methods",
            "Writing every class in a separate programming language",
        ],
        "answer": 0,
        "explanation": "Encapsulation organises state and behaviour together and controls how other code should interact with that state."
    },
    {
        "q": "22. In normal Python convention, what does a name such as `_balance` communicate?",
        "options": [
            "It is non-public/internal by convention, although external access is technically possible",
            "It is encrypted",
            "Python syntax forbids reading it outside the class",
            "It is automatically an abstract property",
        ],
        "answer": 0,
        "explanation": "A single leading underscore is a convention that signals internal/non-public use rather than enforced access control."
    },
    {
        "q": "23. What happens to an attribute named `__pin` inside a class named `Account`?",
        "options": [
            "It is name-mangled to a form similar to `_Account__pin`",
            "It becomes cryptographically encrypted",
            "It is deleted when the constructor finishes",
            "It automatically becomes a class method",
        ],
        "answer": 0,
        "explanation": "Double-leading underscores trigger name mangling, principally to reduce accidental name collisions."
    },
    {
        "q": "24. Which statement about double-leading-underscore attributes is most accurate?",
        "options": [
            "They provide name mangling, not a strong security/privacy boundary",
            "They can never be inspected under any circumstance",
            "They are identical to global variables",
            "They automatically create getters and setters",
        ],
        "answer": 0,
        "explanation": "Name mangling discourages accidental access and collisions but is not a security feature."
    },
    {
        "q": "25. What is a getter?",
        "options": [
            "An interface used to retrieve a value from an object",
            "A method that always deletes an object",
            "A special form of inheritance",
            "A loop that creates classes",
        ],
        "answer": 0,
        "explanation": "A getter returns or exposes a value, potentially through controlled logic."
    },
    {
        "q": "26. What is a setter commonly used for?",
        "options": [
            "Controlling or validating a state change",
            "Displaying the Python version",
            "Creating an abstract base class automatically",
            "Replacing every method with a property",
        ],
        "answer": 0,
        "explanation": "Setters are useful when assignment should be validated or should trigger additional behaviour."
    },
    {
        "q": "27. What does `@property` primarily allow?",
        "options": [
            "Method-based control while exposing attribute-like access syntax",
            "A class to inherit from multiple interpreters",
            "All attributes to become global",
            "Automatic operator overloading for every operator",
        ],
        "answer": 0,
        "explanation": "Properties let code use attribute-like syntax while getter/setter methods execute behind that interface."
    },
    {
        "q": "28. If a property defines a getter but no setter, what is the normal result of assigning to it?",
        "options": [
            "Assignment is not supported through that property",
            "Python creates a setter automatically",
            "The object becomes abstract",
            "The assignment always changes a global variable",
        ],
        "answer": 0,
        "explanation": "Without a corresponding setter, the property is effectively read-only through that interface."
    },
    {
        "q": "29. In the notification example, a loop calls `channel.send(message)` for both email and SMS objects. Which OOP concept is demonstrated most directly by that loop?",
        "options": ["Polymorphism", "Name mangling", "Recursion", "Slicing"],
        "answer": 0,
        "explanation": "The same `send()` operation is used with different concrete runtime object types."
    },
    {
        "q": "30. Which design best combines abstraction and polymorphism?",
        "options": [
            "An abstract `Shape.area()` contract implemented differently by `Circle` and `Rectangle`, then called through one loop",
            "A program containing only global integers",
            "A class with no methods or attributes",
            "A loop that checks every object's exact type before every operation even though all support the same method",
        ],
        "answer": 0,
        "explanation": "The abstract class defines the common contract and the loop uses the different concrete implementations polymorphically."
    },
]

print(f"Loaded {len(quiz_questions)} quiz questions.")

In [ ]:
# Interactive 30-question quiz application
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    state = {
        "index": 0,
        "answers": {},   # question index -> selected option index
        "checked": set(),
    }

    title = widgets.HTML()
    progress = widgets.IntProgress(value=1, min=1, max=len(quiz_questions), description="Question")
    radio = widgets.RadioButtons(options=[], description="", layout=widgets.Layout(width="95%"))
    feedback = widgets.HTML()
    score_html = widgets.HTML()

    check_btn = widgets.Button(description="Check answer", button_style="success")
    prev_btn = widgets.Button(description="Previous")
    next_btn = widgets.Button(description="Next")
    restart_btn = widgets.Button(description="Restart quiz", button_style="warning")

    def score_now():
        score = 0
        for idx in state["checked"]:
            selected = state["answers"].get(idx)
            if selected == quiz_questions[idx]["answer"]:
                score += 1
        return score

    def render():
        idx = state["index"]
        item = quiz_questions[idx]
        title.value = f"<h3>{item['q']}</h3>"
        radio.options = [(text, i) for i, text in enumerate(item["options"])]
        radio.value = state["answers"].get(idx, None)
        progress.value = idx + 1
        prev_btn.disabled = idx == 0
        next_btn.description = "Finish" if idx == len(quiz_questions)-1 else "Next"

        score = score_now()
        score_html.value = f"<b>Checked:</b> {len(state['checked'])}/{len(quiz_questions)} &nbsp; | &nbsp; <b>Score:</b> {score}/{len(state['checked']) if state['checked'] else 0}"

        if idx in state["checked"]:
            selected = state["answers"].get(idx)
            correct = item["answer"]
            if selected == correct:
                status = "✅ Correct"
            else:
                status = f"❌ Not quite. Correct answer: {item['options'][correct]}"
            feedback.value = f"<div style='padding:10px;border:1px solid #999;border-radius:8px;'><b>{status}</b><br>{item['explanation']}</div>"
        else:
            feedback.value = ""

    def remember_selection(change):
        if change.get("name") == "value" and change.get("new") is not None:
            state["answers"][state["index"]] = change["new"]

    def check_answer(_):
        idx = state["index"]
        if radio.value is None:
            feedback.value = "<b>Select an answer first.</b>"
            return
        state["answers"][idx] = radio.value
        state["checked"].add(idx)
        render()

    def previous(_):
        if state["index"] > 0:
            state["index"] -= 1
            render()

    def next_question(_):
        if state["index"] < len(quiz_questions) - 1:
            state["index"] += 1
            render()
        else:
            score = score_now()
            feedback.value = (
                f"<div style='padding:14px;border:2px solid #555;border-radius:10px;'>"
                f"<h3>Quiz complete</h3>"
                f"Checked questions: {len(state['checked'])}/{len(quiz_questions)}<br>"
                f"Final score on checked questions: <b>{score}/{len(state['checked'])}</b>"
                f"</div>"
            )

    def restart(_):
        state["index"] = 0
        state["answers"].clear()
        state["checked"].clear()
        render()

    radio.observe(remember_selection, names="value")
    check_btn.on_click(check_answer)
    prev_btn.on_click(previous)
    next_btn.on_click(next_question)
    restart_btn.on_click(restart)

    render()
    display(widgets.VBox([
        progress,
        title,
        radio,
        widgets.HBox([check_btn, prev_btn, next_btn, restart_btn]),
        feedback,
        score_html,
    ]))
except Exception as exc:
    print("Interactive quiz unavailable:", exc)
    print("The 30 questions remain available in quiz_questions above.")

## 22.1 Text-based quiz fallback and answer key

If widgets are unavailable, you can print every question and its choices using the following cell.

In [ ]:
for i, item in enumerate(quiz_questions, start=1):
    print(item["q"])
    for letter, option in zip("ABCD", item["options"]):
        print(f"  {letter}. {option}")
    print()

<details>
<summary><b>Show 30-question answer key</b></summary>

| Q | Answer | Core idea |
|---:|:---:|---|
| 1 | B | Same interface/operation, different type-appropriate behaviour |
| 2 | B | Runtime is while the program executes |
| 3 | B | Runtime object determines method lookup |
| 4 | A | Method overriding |
| 5 | A | Behaviour/interface matters more than a required parent class |
| 6 | A | Same `move()` call across different objects |
| 7 | A | Special double-underscore method name |
| 8 | B | `__str__` |
| 9 | A | `__repr__` |
| 10 | C | `__len__` |
| 11 | C | `__add__` |
| 12 | C | `__eq__` |
| 13 | A | Class-specific operator behaviour |
| 14 | A | Unsupported operand combination |
| 15 | B | Special value vs exception |
| 16 | A | Essential interface vs implementation details |
| 17 | A | `abc` module |
| 18 | A | Incomplete abstract subclass cannot be instantiated |
| 19 | A | Abstract methods may contain code |
| 20 | A | Runtime exception vs ABC instantiation rule |
| 21 | A | Bundle state/behaviour + controlled interface |
| 22 | A | Non-public by convention |
| 23 | A | Name mangling |
| 24 | A | Not a security boundary |
| 25 | A | Retrieve a value |
| 26 | A | Control/validate state change |
| 27 | A | Attribute-like syntax backed by methods |
| 28 | A | Read-only through that property interface |
| 29 | A | Polymorphism |
| 30 | A | Abstract contract + multiple implementations |

</details>

# 23. Lecture Summary

## Polymorphism
- One interface or operation can work with multiple object types.
- The runtime object determines the behaviour used.
- Method overriding is a common inheritance-based form.
- Python also commonly uses duck typing.

## Dunder / special methods
- Special methods integrate custom classes with Python syntax and protocols.
- Examples: `__str__`, `__repr__`, `__len__`, `__eq__`, `__add__`.
- Operator overloading should have a natural, predictable meaning.

## Abstraction
- Focus on essential behaviour while separating implementation details.
- `ABC` and `@abstractmethod` can define a required interface.
- Incomplete subclasses cannot be instantiated under the ABC mechanism.

## Encapsulation
- Bundle state and behaviour into meaningful objects.
- Public names are intended for use.
- `_name` signals non-public/internal use by convention.
- `__name` is name-mangled, not secured.
- Properties provide controlled, attribute-like access.

### Big picture

```text
Abstraction  → defines what must be possible
Inheritance  → shares/specialises structure and behaviour
Polymorphism → lets one operation work across implementations
Encapsulation→ protects object invariants through a clear interface
```

# 24. Final Reflection

Write short answers in your own words:

1. Why is method overriding considered polymorphic?
2. What is the difference between a variable name and an object's runtime type in Python?
3. When would `__str__` be more useful than the default object representation?
4. Why should `Point.__add__` return a new `Point`?
5. What problem does an abstract base class solve in a larger codebase?
6. Why is `_balance` not truly protected in Python?
7. What does name mangling try to prevent?
8. When is a property better than a plain public attribute?
9. What is the difference between `NotImplemented` and `NotImplementedError`?
10. Give one design of your own that uses all four OOP pillars.

# 25. Further Reading / Documentation

For authoritative Python details, consult the official Python documentation:

- **Python Data Model — special method names and operator behaviour:**  
  https://docs.python.org/3/reference/datamodel.html
- **`abc` — Abstract Base Classes:**  
  https://docs.python.org/3/library/abc.html
- **Built-in `property`:**  
  https://docs.python.org/3/library/functions.html#property
- **Built-in `len`, `repr`, `str`, `isinstance`:**  
  https://docs.python.org/3/library/functions.html

> Course-level explanations in this workbook intentionally emphasise the concepts students need to apply. The official documentation contains additional protocol and implementation details for advanced study.